# BR01C — Recover failed MedGemma reference extractions

Run in the **same MedGemma environment/checkpoint as BR01B** after every eligible image has been attempted. This keeps all successful original outputs and the original extraction journal unchanged. Only originally failed images are retried, under a separate signed protocol: **2048 tokens, repetition penalty 1.15, no-repeat 8-gram**. The image-only prompt is unchanged. A complete output and the existing repetition QC must both pass.

Do not edit BR01B's frozen token budget or helper files. This notebook is an additive exception-recovery route; it does not claim that every image used identical decoding. Preserve its audit for the manuscript. No failed record is removed or relabeled as successful without a new valid generation.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from common import parent_setup
from ecg_external_generation import local_snapshot
from sensor_candidate_generation import load_runtime
MODEL_PATH = None
MODEL_PATH = local_snapshot('google/medgemma-1.5-4b-it', MODEL_PATH)
runtime = load_runtime('cxr', MODEL_PATH)
print('MedGemma revision:', runtime['revision'])


In [ ]:
from extensions.cxr_recovery import recover
RECOVERY_VERSION = 'recovery_t2048_r115_ng8_v1'
MAX_NEW = 2  # Only the two originally failed records are eligible.
RETRY_ERRORS = False
MAX_RUN_HOURS = 2.0  # Below remaining allocation time after model loading.
result = recover(CFG, runtime, recovery_version=RECOVERY_VERSION,
    max_new=MAX_NEW, retry_errors=RETRY_ERRORS, max_run_hours=MAX_RUN_HOURS, max_new_tokens=2048)
print(result)
if result['combined_ok'] != result['total']:
    raise RuntimeError('Recovery incomplete. Inspect failures; do not finalize or accept truncated/repetitive output.')

In [ ]:
from common import latest
RECOVERY_DIR = RUN / 'extraction' / RECOVERY_VERSION
for row in latest(RECOVERY_DIR / 'cxr_candidates.jsonl').values():
    print(row['query_id'], row['status'])
    print(row.get('text', row.get('error')))
    print()
# Inspect these two reports for generation quality before running the final cell.


The final cell below **replaces BR01's original finalization cell** for this recovery route. It validates original successes plus signed recovery results, then creates the normal banks/indexes expected by BR02. The original BR01 finalizer only knows the original journal and will continue to reject the original failures; do not rerun it for this derived bank.

The retry journal is checkpointed per image. After interruption, reload this notebook with the same settings; successful retries are skipped. Keep the native BR01B extraction idle once recovery starts, because its source journal is pinned by hash. If either retry still fails, preserve the error and investigate before changing the protocol/version. No incomplete output is admitted.

In [ ]:
from extensions.cxr_recovery import finalize_recovered_banks
print(finalize_recovered_banks(CFG, recovery_version=RECOVERY_VERSION))
# Require ready=True, then continue to BR02 -> BR03 -> BR03A -> BR04A -> BR05.
